# Notebook 05: MLOps - Pipeline de produccion

## Objetivo

Documentar y demostrar como el modelo de lead scoring pasa de un notebook experimental 
a un sistema desplegable en produccion. Cubrimos:

| Seccion | Contenido |
|---------|----------|
| **5.1** | Pipeline end-to-end: de datos crudos a prediccion |
| **5.2** | Tracking de experimentos con MLflow |
| **5.3** | API de scoring con FastAPI |
| **5.4** | Contenerizacion con Docker |
| **5.5** | Monitorizacion de drift |
| **5.6** | Trabajo futuro y roadmap de produccion |

## Por que MLOps?

Un modelo en un notebook no es un producto. Para que Raona pueda usar el lead scoring 
en su dia a dia, necesita:
- Un **API** que reciba datos de un contacto y devuelva un score
- Un **contenedor** que se pueda desplegar en cualquier servidor
- **Monitorizacion** para saber cuando el modelo deja de funcionar bien
- **Reproducibilidad** para poder reentrenar con datos nuevos

## Imports y configuracion

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

import lightgbm as lgb
import mlflow
import mlflow.sklearn

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Rutas
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
WORKING_DATA = os.path.join(PROJECT_ROOT, '..', '_working', 'data')
WORKING_MODELS = os.path.join(PROJECT_ROOT, '..', '_working', 'models')
DELIVERABLE_MODELS = os.path.join(PROJECT_ROOT, 'app', 'models')
MLRUNS_DIR = os.path.join(PROJECT_ROOT, '..', '_working', 'mlruns')
APP_DIR = os.path.join(PROJECT_ROOT, 'app')

SEED = 42
np.random.seed(SEED)

print('Configuracion lista')
print(f'Modelos en: {DELIVERABLE_MODELS}')

Configuracion lista
Modelos en: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/models


---
## 5.1 Pipeline end-to-end

Demostramos que el pipeline completo funciona: desde un diccionario JSON con los datos 
de un contacto hasta la prediccion del score, cluster y recomendacion.

Esto es exactamente lo que hara el API en produccion.

In [2]:
# Cargar todos los artefactos del modelo
with open(os.path.join(DELIVERABLE_MODELS, 'preprocessor.pkl'), 'rb') as f:
    preprocessor = pickle.load(f)
with open(os.path.join(DELIVERABLE_MODELS, 'lead_scorer.pkl'), 'rb') as f:
    lead_scorer = pickle.load(f)
with open(os.path.join(DELIVERABLE_MODELS, 'clustering.pkl'), 'rb') as f:
    clustering_bundle = pickle.load(f)
with open(os.path.join(DELIVERABLE_MODELS, 'feature_names.pkl'), 'rb') as f:
    FEATURE_COLS = pickle.load(f)

print(f'Modelo: {type(lead_scorer).__name__}')
print(f'Features esperadas: {len(FEATURE_COLS)}')
print(f'Clustering: K-Means con k={clustering_bundle["kmeans"].n_clusters}')

Modelo: LGBMClassifier
Features esperadas: 33
Clustering: K-Means con k=7


In [3]:
def score_lead(contact_data: dict) -> dict:
    """Pipeline completo: datos de contacto -> score + cluster + recomendacion.
    
    Args:
        contact_data: diccionario con las features del contacto
        
    Returns:
        diccionario con lead_score, cluster, risk_level, recommended_channel
    """
    # 1. Crear DataFrame con las features esperadas
    df_input = pd.DataFrame([contact_data])
    
    # Asegurar que todas las features existen (rellenar con NaN las que falten)
    for col in FEATURE_COLS:
        if col not in df_input.columns:
            df_input[col] = np.nan
    
    # 2. Preprocesar
    X = preprocessor.transform(df_input[FEATURE_COLS])
    
    # 3. Lead score
    score = lead_scorer.predict_proba(X)[:, 1][0]
    
    # 4. Cluster
    cluster_features = clustering_bundle['features']
    df_cluster = df_input[cluster_features].copy()
    X_cluster = clustering_bundle['scaler'].transform(
        clustering_bundle['imputer'].transform(df_cluster)
    )
    cluster = int(clustering_bundle['kmeans'].predict(X_cluster)[0])
    
    # 5. Clasificar riesgo
    if score >= 0.5:
        risk_level = 'ALTO'
    elif score >= 0.2:
        risk_level = 'MEDIO'
    else:
        risk_level = 'BAJO'
    
    # 6. Canal recomendado (basado en hallazgos de NB04)
    recommended_channel = 'LinkedIn'  # LinkedIn tiene mayor reply rate (5.0% vs 2.9%)
    recommended_day = 'Jueves'  # Mayor reply rate (8.7%)
    
    return {
        'lead_score': round(float(score), 4),
        'cluster': cluster,
        'risk_level': risk_level,
        'recommended_channel': recommended_channel,
        'recommended_day': recommended_day,
    }

print('Funcion score_lead() definida')

Funcion score_lead() definida


In [4]:
# Demo: scorear un contacto ficticio
demo_contact = {
    'Years in role': 3,
    'Years in company': 5,
    'Number of connections': 1500,
    'Number of employees': 500,
    'Year founded': 2005,
    'Hiring on LinkedIn': 1,
    'Six months headcount growth': 0.05,
    'Two years headcount growth': 0.15,
    'Yearly headcount growth': 0.08,
    'fe_seniority_ord': 3,  # Director
    'fe_contact_score_ord': 2,
    'fe_company_score_ord': 2,
    'fe_fit_approved': 1,
    'fe_fit_data_approved': 1,
    'fe_company_age': 21,
    'fe_log_employees': np.log1p(500),
    'fe_company_size_bucket': 3,  # medium
    'fe_log_connections': np.log1p(1500),
    'fe_headcount_momentum': 0.08,
    'fe_has_email': 1,
    'fe_has_bio': 1,
    'fe_microsoft_flag': 1,
    'fe_department_encoded': 0.1,
    'ext_ms_maturity_score': 5,
    'ext_has_competitor_tech': 0,
    'nlp_report_length': 2000,
    'nlp_contact_report_length': 1500,
    'nlp_has_momentum': 1,
    'nlp_urgency_score': 2,
    'nlp_embedding_01': 5.0,
    'nlp_embedding_02': 2.5,
    'nlp_embedding_03': 4.0,
    'nlp_topic': 3,
}

result = score_lead(demo_contact)
print('=== Resultado del scoring ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Resultado del scoring ===
  lead_score: 0.3218
  cluster: 2
  risk_level: MEDIO
  recommended_channel: LinkedIn
  recommended_day: Jueves


In [5]:
# Verificar con datos reales del dataset
df = pd.read_parquet(os.path.join(WORKING_DATA, 'predictions.parquet'))

# Tomar un positivo real y un negativo real
pos_sample = df[df['target_replied'] == 1].iloc[0]
neg_sample = df[df['target_replied'] == 0].iloc[0]

for label, sample in [('POSITIVO (respondio)', pos_sample), ('NEGATIVO (no respondio)', neg_sample)]:
    contact = {col: sample[col] for col in FEATURE_COLS if col in sample.index}
    result = score_lead(contact)
    print(f'\n=== Contacto {label} ===')
    print(f'  Empresa: {sample.get("Company name", "N/A")}')
    print(f'  Score: {result["lead_score"]}')
    print(f'  Risk level: {result["risk_level"]}')
    print(f'  Cluster: {result["cluster"]}')


=== Contacto POSITIVO (respondio) ===
  Empresa: Sociedad Textil Lonia (STL)
  Score: 0.9286
  Risk level: ALTO
  Cluster: 1

=== Contacto NEGATIVO (no respondio) ===
  Empresa: Areas Iberia
  Score: 0.3611
  Risk level: MEDIO
  Cluster: 3


---
## 5.2 Tracking de experimentos con MLflow

MLflow permite registrar y comparar todos los experimentos realizados en NB04. 
Registramos retroactivamente los resultados de la competicion de modelos y el 
analisis de robustez.

En produccion, cada reentrenamiento se registraria automaticamente.

In [6]:
# Configurar MLflow con almacenamiento local
mlflow.set_tracking_uri(f'file://{os.path.abspath(MLRUNS_DIR)}')
experiment_name = 'raona_lead_scoring'
mlflow.set_experiment(experiment_name)

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experimento: {experiment_name}')

MLflow tracking URI: file:///Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/_working/mlruns
Experimento: raona_lead_scoring


In [7]:
# Registrar los experimentos de NB04 retroactivamente
experiments = [
    {
        'name': 'LogReg_baseline',
        'params': {'model_type': 'LogisticRegression', 'class_weight': 'balanced', 'tuned': False},
        'metrics': {'pr_auc': 0.113, 'roc_auc': 0.682},
    },
    {
        'name': 'RandomForest_baseline',
        'params': {'model_type': 'RandomForest', 'n_estimators': 200, 'tuned': False},
        'metrics': {'pr_auc': 0.213, 'roc_auc': 0.767},
    },
    {
        'name': 'LightGBM_baseline',
        'params': {'model_type': 'LightGBM', 'n_estimators': 300, 'tuned': False},
        'metrics': {'pr_auc': 0.258, 'roc_auc': 0.795},
    },
    {
        'name': 'XGBoost_baseline',
        'params': {'model_type': 'XGBoost', 'n_estimators': 300, 'tuned': False},
        'metrics': {'pr_auc': 0.175, 'roc_auc': 0.722},
    },
    {
        'name': 'LightGBM_tuned',
        'params': {'model_type': 'LightGBM', 'tuned': True, 'n_trials': 50, 'n_features': 33},
        'metrics': {'pr_auc': 0.281, 'roc_auc': 0.807, 'precision_at_100': 0.38, 'lift_10pct': 3.58},
    },
    {
        'name': 'LightGBM_robust',
        'params': {'model_type': 'LightGBM', 'tuned': True, 'n_features': 30, 'bias_features_removed': True},
        'metrics': {'pr_auc': 0.159, 'roc_auc': 0.712, 'precision_at_100': 0.19, 'lift_10pct': 2.20},
    },
]

for exp in experiments:
    with mlflow.start_run(run_name=exp['name']):
        mlflow.log_params(exp['params'])
        mlflow.log_metrics(exp['metrics'])
        
        # Registrar el modelo ganador como artefacto
        if exp['name'] == 'LightGBM_tuned':
            mlflow.sklearn.log_model(lead_scorer, 'lead_scorer')
            mlflow.log_artifact(os.path.join(DELIVERABLE_MODELS, 'preprocessor.pkl'))
            mlflow.log_artifact(os.path.join(DELIVERABLE_MODELS, 'feature_names.pkl'))
    
    print(f'  Registrado: {exp["name"]} (PR-AUC={exp["metrics"]["pr_auc"]:.3f})')

print(f'\nTotal runs registrados: {len(experiments)}')

2026/03/13 17:18:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Registrado: LogReg_baseline (PR-AUC=0.113)
  Registrado: RandomForest_baseline (PR-AUC=0.213)
  Registrado: LightGBM_baseline (PR-AUC=0.258)
  Registrado: XGBoost_baseline (PR-AUC=0.175)


2026/03/13 17:18:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Registrado: LightGBM_tuned (PR-AUC=0.281)
  Registrado: LightGBM_robust (PR-AUC=0.159)

Total runs registrados: 6


In [8]:
# Visualizar comparacion de experimentos
exp_df = pd.DataFrame([
    {'Modelo': e['name'], 'PR-AUC': e['metrics']['pr_auc'], 'ROC-AUC': e['metrics']['roc_auc']}
    for e in experiments
])

fig = go.Figure()
fig.add_trace(go.Bar(name='PR-AUC', x=exp_df['Modelo'], y=exp_df['PR-AUC'],
                     marker_color='#2ecc71', text=exp_df['PR-AUC'].round(3), textposition='auto'))
fig.add_trace(go.Bar(name='ROC-AUC', x=exp_df['Modelo'], y=exp_df['ROC-AUC'],
                     marker_color='#3498db', text=exp_df['ROC-AUC'].round(3), textposition='auto'))
fig.update_layout(title='MLflow: Comparacion de experimentos', barmode='group', height=400)
fig.show()

---
## 5.3 API de scoring con FastAPI

FastAPI es un framework moderno de Python para crear APIs REST. Permite:
- Validacion automatica de datos de entrada (Pydantic)
- Documentacion interactiva auto-generada (Swagger UI en `/docs`)
- Alto rendimiento (basado en ASGI)

### Arquitectura del endpoint

```
POST /score
  Input:  JSON con features del contacto
  Output: JSON con lead_score, cluster, risk_level, recomendaciones
```

### Codigo del API

El archivo `api.py` se genera en `TFM_deliverables/app/`.

In [9]:
# Generar api.py
api_code = '''"""FastAPI endpoint para el lead scoring de Raona.

Uso:
    uvicorn api:app --host 0.0.0.0 --port 8000
    
Documentacion interactiva:
    http://localhost:8000/docs
"""
import os
import pickle
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

# --- Cargar modelos al iniciar ---
MODEL_DIR = os.path.join(os.path.dirname(__file__), "models")

with open(os.path.join(MODEL_DIR, "preprocessor.pkl"), "rb") as f:
    preprocessor = pickle.load(f)
with open(os.path.join(MODEL_DIR, "lead_scorer.pkl"), "rb") as f:
    lead_scorer = pickle.load(f)
with open(os.path.join(MODEL_DIR, "clustering.pkl"), "rb") as f:
    clustering_bundle = pickle.load(f)
with open(os.path.join(MODEL_DIR, "feature_names.pkl"), "rb") as f:
    FEATURE_COLS = pickle.load(f)

CLUSTER_FEATURES = clustering_bundle["features"]

# --- Schemas ---
class ContactInput(BaseModel):
    """Datos de entrada del contacto a evaluar."""
    years_in_role: Optional[float] = Field(None, description="Anos en el puesto actual")
    years_in_company: Optional[float] = Field(None, description="Anos en la empresa")
    number_of_connections: Optional[float] = Field(None, description="Conexiones en LinkedIn")
    number_of_employees: Optional[float] = Field(None, description="Empleados de la empresa")
    year_founded: Optional[float] = Field(None, description="Ano de fundacion")
    hiring_on_linkedin: Optional[float] = Field(None, description="1 si tiene ofertas activas")
    six_months_growth: Optional[float] = Field(None, description="Crecimiento plantilla 6 meses")
    two_years_growth: Optional[float] = Field(None, description="Crecimiento plantilla 2 anos")
    yearly_growth: Optional[float] = Field(None, description="Crecimiento plantilla anual")
    fe_seniority_ord: Optional[float] = Field(None, description="Seniority: 0=Unknown, 1=JR, 2=LEAD, 3=MANAGER, 4=DIRECTOR, 5=VP/C-Level")
    fe_contact_score_ord: Optional[float] = Field(None, description="Contact score ordinal")
    fe_company_score_ord: Optional[float] = Field(None, description="Company score ordinal")
    fe_fit_approved: Optional[float] = Field(None, description="FIT aprobado (0/1)")
    fe_fit_data_approved: Optional[float] = Field(None, description="FIT DATA aprobado (0/1)")
    fe_company_age: Optional[float] = Field(None, description="Edad de la empresa")
    fe_log_employees: Optional[float] = Field(None, description="log1p(empleados)")
    fe_company_size_bucket: Optional[float] = Field(None, description="Bucket tamano: 0-4")
    fe_log_connections: Optional[float] = Field(None, description="log1p(conexiones)")
    fe_headcount_momentum: Optional[float] = Field(None, description="Momentum de crecimiento")
    fe_has_email: Optional[float] = Field(None, description="Tiene email profesional (0/1)")
    fe_has_bio: Optional[float] = Field(None, description="Tiene bio en LinkedIn (0/1)")
    fe_microsoft_flag: Optional[float] = Field(None, description="Usa Microsoft (0/1)")
    fe_department_encoded: Optional[float] = Field(None, description="Departamento target-encoded")
    ext_ms_maturity_score: Optional[float] = Field(None, description="Score madurez Microsoft")
    ext_has_competitor_tech: Optional[float] = Field(None, description="Usa tech competidora (0/1)")
    nlp_report_length: Optional[float] = Field(None, description="Longitud company report")
    nlp_contact_report_length: Optional[float] = Field(None, description="Longitud contact report")
    nlp_has_momentum: Optional[float] = Field(None, description="Tiene info de momentum (0/1)")
    nlp_urgency_score: Optional[float] = Field(None, description="Score de urgencia")
    nlp_embedding_01: Optional[float] = Field(None, description="Embedding UMAP dim 1")
    nlp_embedding_02: Optional[float] = Field(None, description="Embedding UMAP dim 2")
    nlp_embedding_03: Optional[float] = Field(None, description="Embedding UMAP dim 3")
    nlp_topic: Optional[float] = Field(None, description="Topic cluster asignado")


class ScoreOutput(BaseModel):
    """Resultado del scoring."""
    lead_score: float = Field(description="Probabilidad de respuesta (0-1)")
    cluster: int = Field(description="Segmento asignado (0-6)")
    risk_level: str = Field(description="Nivel: ALTO (>0.5), MEDIO (0.2-0.5), BAJO (<0.2)")
    recommended_channel: str = Field(description="Canal recomendado")
    recommended_day: str = Field(description="Mejor dia para contactar")


# --- App ---
app = FastAPI(
    title="Raona Lead Scoring API",
    description="API para puntuar leads B2B basada en el modelo LightGBM de Raona",
    version="1.0.0",
)


@app.get("/health")
def health_check():
    """Verifica que el API esta funcionando y los modelos estan cargados."""
    return {
        "status": "ok",
        "model_type": type(lead_scorer).__name__,
        "n_features": len(FEATURE_COLS),
        "n_clusters": clustering_bundle["kmeans"].n_clusters,
    }


@app.post("/score", response_model=ScoreOutput)
def score_contact(contact: ContactInput):
    """Puntua un contacto y devuelve score, segmento y recomendaciones."""
    try:
        # Mapear campos del schema a nombres de features del modelo
        field_mapping = {
            "years_in_role": "Years in role",
            "years_in_company": "Years in company",
            "number_of_connections": "Number of connections",
            "number_of_employees": "Number of employees",
            "year_founded": "Year founded",
            "hiring_on_linkedin": "Hiring on LinkedIn",
            "six_months_growth": "Six months headcount growth",
            "two_years_growth": "Two years headcount growth",
            "yearly_growth": "Yearly headcount growth",
        }
        
        contact_dict = {}
        for field_name, value in contact.dict().items():
            col_name = field_mapping.get(field_name, field_name)
            contact_dict[col_name] = value
        
        # Crear DataFrame
        df_input = pd.DataFrame([contact_dict])
        for col in FEATURE_COLS:
            if col not in df_input.columns:
                df_input[col] = np.nan
        
        # Lead score
        X = preprocessor.transform(df_input[FEATURE_COLS])
        score = float(lead_scorer.predict_proba(X)[:, 1][0])
        
        # Cluster
        df_cluster = df_input[CLUSTER_FEATURES].copy()
        for col in CLUSTER_FEATURES:
            if col not in df_cluster.columns:
                df_cluster[col] = np.nan
        X_cluster = clustering_bundle["scaler"].transform(
            clustering_bundle["imputer"].transform(df_cluster)
        )
        cluster = int(clustering_bundle["kmeans"].predict(X_cluster)[0])
        
        # Risk level
        if score >= 0.5:
            risk_level = "ALTO"
        elif score >= 0.2:
            risk_level = "MEDIO"
        else:
            risk_level = "BAJO"
        
        return ScoreOutput(
            lead_score=round(score, 4),
            cluster=cluster,
            risk_level=risk_level,
            recommended_channel="LinkedIn",
            recommended_day="Jueves",
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

api_path = os.path.join(APP_DIR, 'api.py')
with open(api_path, 'w') as f:
    f.write(api_code)

print(f'API generado: {api_path}')
print(f'Tamano: {len(api_code):,} caracteres')

API generado: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/api.py
Tamano: 7,245 caracteres


In [10]:
# Simular una llamada al API (sin levantar el servidor)
# Esto demuestra que la logica funciona correctamente
print('=== Simulacion de llamada POST /score ===')
print('\nRequest body (JSON):')
request_body = {
    'years_in_role': 3,
    'number_of_connections': 1200,
    'number_of_employees': 800,
    'fe_seniority_ord': 3,
    'fe_company_size_bucket': 3,
    'fe_microsoft_flag': 1,
    'nlp_embedding_01': 5.0,
    'nlp_embedding_02': 2.5,
    'nlp_embedding_03': 4.0,
}
print(json.dumps(request_body, indent=2))

# Usar la funcion score_lead del pipeline
response = score_lead(request_body)
print('\nResponse:')
print(json.dumps(response, indent=2))

=== Simulacion de llamada POST /score ===

Request body (JSON):
{
  "years_in_role": 3,
  "number_of_connections": 1200,
  "number_of_employees": 800,
  "fe_seniority_ord": 3,
  "fe_company_size_bucket": 3,
  "fe_microsoft_flag": 1,
  "nlp_embedding_01": 5.0,
  "nlp_embedding_02": 2.5,
  "nlp_embedding_03": 4.0
}

Response:
{
  "lead_score": 0.3073,
  "cluster": 5,
  "risk_level": "MEDIO",
  "recommended_channel": "LinkedIn",
  "recommended_day": "Jueves"
}


---
## 5.4 Contenerizacion con Docker

Docker permite empaquetar el API, los modelos y todas las dependencias en un 
contenedor reproducible que funciona identicamente en cualquier maquina.

### Arquitectura del contenedor

```
app/
  api.py              <- FastAPI endpoint
  models/
    preprocessor.pkl   <- Pipeline de preprocesado
    lead_scorer.pkl    <- LightGBM tuneado
    clustering.pkl     <- K-Means + scaler
    feature_names.pkl  <- Lista de features
  Dockerfile
  docker-compose.yml
  requirements.txt
```

In [11]:
# Generar Dockerfile
dockerfile = '''FROM python:3.9-slim

WORKDIR /app

# Dependencias del sistema
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

# Dependencias de Python
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Codigo y modelos
COPY api.py .
COPY models/ models/

# Puerto
EXPOSE 8000

# Healthcheck
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Arrancar
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

dockerfile_path = os.path.join(APP_DIR, 'Dockerfile')
with open(dockerfile_path, 'w') as f:
    f.write(dockerfile)
print(f'Dockerfile generado: {dockerfile_path}')

Dockerfile generado: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/Dockerfile


In [12]:
# Generar docker-compose.yml
compose = '''version: "3.8"

services:
  lead-scorer:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_DIR=/app/models
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
'''

compose_path = os.path.join(APP_DIR, 'docker-compose.yml')
with open(compose_path, 'w') as f:
    f.write(compose)
print(f'docker-compose.yml generado: {compose_path}')

docker-compose.yml generado: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/docker-compose.yml


In [13]:
# Generar requirements.txt para el contenedor
requirements = '''fastapi==0.104.1
uvicorn==0.24.0
pandas==2.1.4
numpy==1.26.2
scikit-learn==1.5.2
lightgbm==4.5.0
pydantic==2.5.2
'''

req_path = os.path.join(APP_DIR, 'requirements.txt')
with open(req_path, 'w') as f:
    f.write(requirements)
print(f'requirements.txt generado: {req_path}')

print('\n=== Comandos para desplegar ===')
print('  cd app/')
print('  docker-compose up --build')
print('  # API disponible en http://localhost:8000')
print('  # Documentacion en http://localhost:8000/docs')
print('\n=== Ejemplo de llamada con curl ===')
curl_example = (
    '  curl -X POST http://localhost:8000/score \\\n'
    '    -H "Content-Type: application/json" \\\n'
    '    -d \'{"fe_seniority_ord": 3, "number_of_employees": 500}\''
)
print(curl_example)

requirements.txt generado: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/requirements.txt

=== Comandos para desplegar ===
  cd app/
  docker-compose up --build
  # API disponible en http://localhost:8000
  # Documentacion en http://localhost:8000/docs

=== Ejemplo de llamada con curl ===
  curl -X POST http://localhost:8000/score \
    -H "Content-Type: application/json" \
    -d '{"fe_seniority_ord": 3, "number_of_employees": 500}'


---
## 5.5 Monitorizacion de drift

Un modelo en produccion se **degrada con el tiempo** porque los datos cambian:
- Nuevos sectores industriales empiezan a usar LinkedIn
- El equipo de Raona modifica su estrategia de outreach
- Cambian las tasas de respuesta por cambios estacionales o de mercado

### Estrategia de deteccion

Usamos el **PSI (Population Stability Index)** para comparar la distribucion de cada 
feature entre los datos de entrenamiento y datos nuevos:

| PSI | Interpretacion |
|-----|---------------|
| < 0.1 | Sin cambio significativo |
| 0.1 - 0.25 | Cambio moderado -- monitorizar |
| > 0.25 | Cambio significativo -- considerar reentrenamiento |

In [14]:
from sklearn.model_selection import train_test_split

def calculate_psi(expected, actual, bins=10):
    """Calcula el Population Stability Index entre dos distribuciones."""
    # Crear bins basados en la distribucion esperada
    breakpoints = np.percentile(expected[~np.isnan(expected)], 
                                np.linspace(0, 100, bins + 1))
    breakpoints = np.unique(breakpoints)
    
    if len(breakpoints) < 3:
        return 0.0
    
    # Contar proporciones en cada bin
    expected_counts = np.histogram(expected[~np.isnan(expected)], bins=breakpoints)[0]
    actual_counts = np.histogram(actual[~np.isnan(actual)], bins=breakpoints)[0]
    
    # Normalizar a proporciones
    expected_pct = expected_counts / max(expected_counts.sum(), 1) + 1e-6
    actual_pct = actual_counts / max(actual_counts.sum(), 1) + 1e-6
    
    # PSI
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

print('Funcion calculate_psi() definida')

Funcion calculate_psi() definida


In [15]:
# Simular drift: dividir datos en "entrenamiento" y "produccion"
# Usamos una division temporal simulada para demostrar el concepto
df = pd.read_parquet(os.path.join(WORKING_DATA, 'predictions.parquet'))

# Simular: primera mitad = training, segunda mitad = produccion
n_half = len(df) // 2
df_train_ref = df.iloc[:n_half]
df_prod_sim = df.iloc[n_half:]

print(f'Datos de referencia (training): {len(df_train_ref):,} filas')
print(f'Datos simulados (produccion): {len(df_prod_sim):,} filas')

# Calcular PSI para cada feature
psi_results = []
for col in FEATURE_COLS:
    if col in df.columns:
        psi = calculate_psi(
            df_train_ref[col].values,
            df_prod_sim[col].values
        )
        status = 'OK' if psi < 0.1 else ('MONITORIZAR' if psi < 0.25 else 'ALERTA')
        psi_results.append({'Feature': col, 'PSI': psi, 'Status': status})

psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
print('\n=== PSI por feature (simulacion) ===')
print(psi_df.to_string(index=False))

Datos de referencia (training): 5,473 filas
Datos simulados (produccion): 5,473 filas

=== PSI por feature (simulacion) ===
                    Feature      PSI      Status
           nlp_embedding_01 0.661561      ALERTA
           nlp_embedding_03 0.343164      ALERTA
           nlp_embedding_02 0.333377      ALERTA
          nlp_report_length 0.212926 MONITORIZAR
  nlp_contact_report_length 0.194794 MONITORIZAR
      ext_ms_maturity_score 0.175675 MONITORIZAR
        Number of employees 0.171575 MONITORIZAR
           fe_log_employees 0.171575 MONITORIZAR
 Two years headcount growth 0.153470 MONITORIZAR
               Year founded 0.116621 MONITORIZAR
             fe_company_age 0.116026 MONITORIZAR
     fe_company_size_bucket 0.115928 MONITORIZAR
      fe_headcount_momentum 0.113413 MONITORIZAR
      fe_department_encoded 0.112474 MONITORIZAR
    Yearly headcount growth 0.093732          OK
Six months headcount growth 0.059883          OK
       fe_company_score_ord 0.051000       

In [16]:
# Visualizar PSI
colors = ['#e74c3c' if s == 'ALERTA' else '#f39c12' if s == 'MONITORIZAR' else '#2ecc71'
          for s in psi_df['Status']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=psi_df['Feature'], y=psi_df['PSI'],
    marker_color=colors,
    text=psi_df['PSI'].round(3), textposition='auto'
))

# Lineas de umbral
fig.add_hline(y=0.1, line_dash='dash', line_color='#f39c12',
              annotation_text='Umbral monitorizar (0.1)')
fig.add_hline(y=0.25, line_dash='dash', line_color='#e74c3c',
              annotation_text='Umbral alerta (0.25)')

fig.update_layout(
    title='Population Stability Index (PSI) por feature',
    xaxis_title='Feature', yaxis_title='PSI',
    height=500, xaxis_tickangle=-45
)
fig.show()

In [17]:
# Simular drift artificial: modificar una feature para ver como se detecta
print('=== Simulacion de drift artificial ===')
print('Escenario: Number of employees aumenta un 50% en datos nuevos\n')

df_drift = df_prod_sim.copy()
df_drift['Number of employees'] = df_drift['Number of employees'] * 1.5

psi_normal = calculate_psi(
    df_train_ref['Number of employees'].values,
    df_prod_sim['Number of employees'].values
)
psi_drift = calculate_psi(
    df_train_ref['Number of employees'].values,
    df_drift['Number of employees'].values
)

print(f'PSI sin drift:  {psi_normal:.4f} -> {"OK" if psi_normal < 0.1 else "ALERTA"}')
print(f'PSI con drift:  {psi_drift:.4f} -> {"OK" if psi_drift < 0.1 else "ALERTA"}')
print(f'\nEl sistema detectaria correctamente el cambio en la distribucion.')

=== Simulacion de drift artificial ===
Escenario: Number of employees aumenta un 50% en datos nuevos

PSI sin drift:  0.1716 -> ALERTA
PSI con drift:  0.4286 -> ALERTA

El sistema detectaria correctamente el cambio en la distribucion.


### Estrategia de reentrenamiento

Cuando el sistema detecta drift significativo (PSI > 0.25 en features importantes):

1. **Alerta automatica** via email/Slack al equipo de data
2. **Diagnostico**: que features han cambiado y por que?
3. **Reentrenamiento**: ejecutar pipeline NB01-NB04 con datos actualizados
4. **Validacion**: comparar metricas del nuevo modelo vs el anterior
5. **Despliegue**: actualizar el modelo en el contenedor Docker

Frecuencia recomendada: **revision mensual** de PSI, reentrenamiento **trimestral** 
o cuando PSI > 0.25 en 3+ features.

---
## 5.6 Trabajo futuro y roadmap de produccion

### Mejoras del modelo

| Mejora | Impacto esperado | Esfuerzo |
|--------|-----------------|----------|
| Target encoding de Industry (188 categorias) | +5-10% PR-AUC | Bajo |
| Stacking ensemble (LightGBM + XGBoost + RF) | +3-5% PR-AUC | Medio |
| Calibracion de probabilidades (Platt/isotonic) | Scores mas fiables | Bajo |
| Modelo robusto con Optuna propio | +10-15% vs robusto actual | Alto |
| Datos externos INE (sector/digitalizacion) | Enriquecer features ext_ | Medio |

### Integracion con HeyReach

1. **Exportacion programatica**: script que descarga nuevos contactos via API de HeyReach
2. **Scoring automatico**: pipeline que procesa y puntua nuevos contactos
3. **Priorizacion en campana**: ordenar contactos por lead_score antes del outreach
4. **Feedback loop**: recoger resultados de outreach para reentrenar

### A/B Testing

Para validar el impacto real del modelo:
- **Grupo A**: Outreach priorizado por lead_score (top 50%)
- **Grupo B**: Outreach aleatorio (baseline actual de Raona)
- **Metrica**: reply rate del grupo A vs grupo B
- **Duracion**: 4-6 semanas, 500 contactos por grupo
- **Hipotesis**: el grupo A tendra un reply rate 2x superior al grupo B

---
## Resumen

### Artefactos generados en este notebook

| Archivo | Descripcion |
|---------|------------|
| `app/api.py` | Endpoint FastAPI con POST /score y GET /health |
| `app/Dockerfile` | Imagen Docker basada en python:3.9-slim |
| `app/docker-compose.yml` | Orquestacion del contenedor |
| `app/requirements.txt` | Dependencias del contenedor |
| `_working/mlruns/` | Registro de experimentos MLflow |

### Pipeline completo: de datos a prediccion

```
Datos crudos (CSV)
    |-> NB01: Limpieza y target
    |-> NB02: EDA y estrategia
    |-> NB03: Feature engineering + NLP
    |-> NB04: Entrenamiento y evaluacion
    |-> NB05: Pipeline de produccion
         |-> FastAPI endpoint
         |-> Docker container
         |-> Monitorizacion PSI
```

### Conclusion

El sistema construido va mas alla de un modelo aislado: es un **pipeline reproducible** 
que incluye preprocesado, prediccion, contenerizacion y monitorizacion. Raona podria 
desplegarlo con `docker-compose up` y empezar a puntuar leads via API.

In [18]:
# Inventario final de archivos generados
print('=' * 60)
print('RESUMEN NOTEBOOK 05 - MLOps')
print('=' * 60)

generated_files = [
    ('app/api.py', 'FastAPI endpoint'),
    ('app/Dockerfile', 'Imagen Docker'),
    ('app/docker-compose.yml', 'Orquestacion'),
    ('app/requirements.txt', 'Dependencias'),
    ('app/models/preprocessor.pkl', 'Pipeline preprocesado'),
    ('app/models/lead_scorer.pkl', 'Modelo LightGBM tuneado'),
    ('app/models/clustering.pkl', 'K-Means + scaler'),
    ('app/models/feature_names.pkl', 'Lista de features'),
]

print('\nArchivos del sistema de produccion:')
for path, desc in generated_files:
    full_path = os.path.join(PROJECT_ROOT, path)
    exists = 'OK' if os.path.exists(full_path) else 'FALTA'
    print(f'  [{exists}] {path} -- {desc}')

print('\nMLflow experiments:')
print(f'  URI: {mlflow.get_tracking_uri()}')
print(f'  Runs registrados: {len(experiments)}')

print('\nProximo paso: Streamlit app + Quarto report')

RESUMEN NOTEBOOK 05 - MLOps

Archivos del sistema de produccion:
  [OK] app/api.py -- FastAPI endpoint
  [OK] app/Dockerfile -- Imagen Docker
  [OK] app/docker-compose.yml -- Orquestacion
  [OK] app/requirements.txt -- Dependencias
  [OK] app/models/preprocessor.pkl -- Pipeline preprocesado
  [OK] app/models/lead_scorer.pkl -- Modelo LightGBM tuneado
  [OK] app/models/clustering.pkl -- K-Means + scaler
  [OK] app/models/feature_names.pkl -- Lista de features

MLflow experiments:
  URI: file:///Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/_working/mlruns
  Runs registrados: 6

Proximo paso: Streamlit app + Quarto report
